In [0]:
df = spark.read.option("header", True) \
    .option("sep", ";") \
    .csv("/Volumes/workspace/default/dados/Carros.csv")
df.write.mode("overwrite").saveAsTable("carros")

In [0]:
%sql
select * from carros

Consumo,Cilindros,Cilindradas,RelEixoTraseiro,Peso,Tempo,TipoMotor,Transmissao,Marchas,Carburadors,HP
21,6,160,39,262,1646,0,1,4,4,110
21,6,160,39,2875,1702,0,1,4,4,110
228,4,108,385,232,1861,1,1,4,1,93
214,6,258,308,3215,1944,1,0,3,1,110
187,8,360,315,344,1702,0,0,3,2,175
181,6,225,276,346,2022,1,0,3,1,105
143,8,360,321,357,1584,0,0,3,4,245
244,4,1467,369,319,20,1,0,4,2,62
228,4,1408,392,315,229,1,0,4,2,95
192,6,1676,392,344,183,1,0,4,4,123


In [0]:
%sql
SELECT
    *
FROM
    carros
WHERE
    cilindros > 4

Consumo,Cilindros,Cilindradas,RelEixoTraseiro,Peso,Tempo,TipoMotor,Transmissao,Marchas,Carburadors,HP
21,6,160,39,262,1646,0,1,4,4,110
21,6,160,39,2875,1702,0,1,4,4,110
214,6,258,308,3215,1944,1,0,3,1,110
187,8,360,315,344,1702,0,0,3,2,175
181,6,225,276,346,2022,1,0,3,1,105
143,8,360,321,357,1584,0,0,3,4,245
192,6,1676,392,344,183,1,0,4,4,123
178,6,1676,392,344,189,1,0,4,4,123
164,8,2758,307,407,174,0,0,3,3,180
173,8,2758,307,373,176,0,0,3,3,180


In [0]:
%sql
SELECT
    TipoMotor,
    AVG(Peso) as media_peso,
    count(*) as Quantidade
FROM
    carros
WHERE
    Cilindros = 6
GROUP BY
    TipoMotor
HAVING AVG(Peso) >= 100
ORDER BY
    Quantidade DESC
    


TipoMotor,media_peso,Quantidade
1,1062.25,4
0,1138.0,3


In [0]:
df_carros = spark.table("carros")

df_carros_filtrados = df_carros.filter(df_carros["cilindros"] > 6)

df_carros_filtrados.show()

+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|Consumo|Cilindros|Cilindradas|RelEixoTraseiro|Peso|Tempo|TipoMotor|Transmissao|Marchas|Carburadors| HP|
+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|    187|        8|        360|            315| 344| 1702|        0|          0|      3|          2|175|
|    143|        8|        360|            321| 357| 1584|        0|          0|      3|          4|245|
|    164|        8|       2758|            307| 407|  174|        0|          0|      3|          3|180|
|    173|        8|       2758|            307| 373|  176|        0|          0|      3|          3|180|
|    152|        8|       2758|            307| 378|   18|        0|          0|      3|          3|180|
|    104|        8|        472|            293| 525| 1798|        0|          0|      3|          4|205|
|    104|        8|        460|              3|5424| 17

Executando comando SQL para filtrar os carros com cilindro igual 6, usando spark sql

In [0]:
df_carros_filtrados = spark.sql("SELECT * FROM carros WHERE cilindros = 6")

df_carros_filtrados.show()

+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|Consumo|Cilindros|Cilindradas|RelEixoTraseiro|Peso|Tempo|TipoMotor|Transmissao|Marchas|Carburadors| HP|
+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|     21|        6|        160|             39| 262| 1646|        0|          1|      4|          4|110|
|     21|        6|        160|             39|2875| 1702|        0|          1|      4|          4|110|
|    214|        6|        258|            308|3215| 1944|        1|          0|      3|          1|110|
|    181|        6|        225|            276| 346| 2022|        1|          0|      3|          1|105|
|    192|        6|       1676|            392| 344|  183|        1|          0|      4|          4|123|
|    178|        6|       1676|            392| 344|  189|        1|          0|      4|          4|123|
|    197|        6|        145|            362| 277|  1

In [0]:
df_resultado = spark.sql("""
    SELECT TipoMotor,
    AVG(PESO) as peso_medio,
    COUNT(*) AS qtd
    FROM carros
    WHERE cilindros = 6
    GROUP BY TipoMotor
    HAVING avg(peso) > 100
    ORDER BY qtd desc
""")

df_resultado.display()

TipoMotor,peso_medio,qtd
1,1062.25,4
0,1138.0,3


Usando Pyspark

In [0]:
from pyspark.sql.functions import avg, count, desc
df_resultado = (
    spark.table("carros")
    .filter("cilindros = 6")
    .groupBy("TipoMotor")
    .agg(
        avg("Peso").alias("peso_medio"),
        count("*").alias("quantidade")
    )
    .filter("avg(Peso) > 100")
    .orderBy(desc("Quantidade"))
)

df_resultado.display()

TipoMotor,peso_medio,quantidade
1,1062.25,4
0,1138.0,3
